# Lekcia 13 - Pamäť agenta s Cognee znalosťovými grafmi


## Nastavenie

Tento notebook demonštruje, ako vytvoriť inteligentného **kódovacieho asistenta** s perzistentnou pamäťou pomocou znalostných grafov [**Cognee**](https://www.cognee.ai/) a **Microsoft Agent Frameworku** (MAF).

Cognee premieňa nestrukturovaný text na štruktúrovaný znalostný graf s možnosťou dotazovania, podporený vektorovými embeddingmi — poskytuje vášmu agentovi bohatú dlhodobú pamäť s uvedomením si vzťahov.

### Čo sa naučíte
1. **Vytvárať znalostné grafy**: Premeniť profily vývojárov a osvedčené postupy na štruktúrované, dotazovateľné znalosti.
2. **Integrovať Cognee s MAF**: Použiť `@tool` funkcie na umoženie MAF agentovi dotazovať sa v znalostnom grafe Cognee.
3. **Konverzácie s uvedomením relácie**: Udržiavať kontext cez viacero otázok v rovnakej relácii.
4. **Dlhodobá pamäť**: Perzistovať dôležité znalosti cez relácie a vyvolať ich v nových konverzáciách.

### Požiadavky
- Python 3.9+
- Lokálne bežiaci Redis (`docker run -d -p 6379:6379 redis`) pre správu relácií
- API kľúč pre LLM (napríklad OpenAI) — nastaviť `LLM_API_KEY` v `.env`
- `CACHING=true` v `.env` (vyžaduje sa pre Cognee relácie)
- Projekt Azure AI Foundry s nasadeným chat modelom
- `AZURE_AI_PROJECT_ENDPOINT` a `AZURE_AI_MODEL_DEPLOYMENT_NAME` v `.env`
- Autentifikovaný Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy pamäte agenta

Tento notebook skúma rovnaké tri typy pamäte ako v hlavnom notebooku Lekcie 13, ale používa Cognee ako backend pre dlhodobú pamäť:

| Typ pamäte | Mechanizmus | Životnosť |
|---|---|---|
| **Pracovná** | `agent.create_session()` (MAF) | Jediný rozhovor |
| **Krátkodobá** | Cache relácie Cognee (Redis) | Jedna relácia |
| **Dlhodobá** | Cognee znalostný graf + vektory | Trvalá |

### Architektúra pamäte Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Pripraviť Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Časť 1 — Budovanie znalostnej databázy

Získavame tri druhy údajov na vytvorenie komplexnej znalostnej databázy pre nášho kódovacieho asistenta:

1. **Profily vývojárov** — osobné odborné znalosti a technické skúsenosti
2. **Najlepšie praktiky Pythonu** — Zen Pythonu s praktickými usmerneniami
3. **Historické rozhovory** — minulé Q&A relácie medzi vývojármi a AI asistentmi


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizácia znalostného grafu

Cognee môže vytvoriť interaktívnu HTML vizualizáciu entít a vzťahov, ktoré extrahoval.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obohacujte pamäť pomocou Memify

`memify()` analyzuje znalostný graf a generuje inteligentné pravidlá — identifikuje vzory, osvedčené postupy a vzťahy medzi konceptmi.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Časť 2 — MAF Agent s nástrojmi Cognee

Teraz vytvoríme MAF agenta, ktorý môže dotazovať vedomostný graf Cognee prostredníctvom funkcií `@tool`. Toto umožňuje agentovi využiť plnú silu grafovo orientovaného sémantického vyhľadávania pri zachovaní konverzačného kontextu cez relácie.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Pracovná pamäť so sessions

`AgentSession` (vytvorený cez `agent.create_session()`) poskytuje pracovnú pamäť v rámci session. Agent sa môže odvolávať na skoršie správy a zároveň dotazovať sa v dlhodobej znalostnej sieti Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nová relácia — Dlhodobá pamäť pretrváva

Spustenie novej relácie vymaže pracovnú pamäť, no graf vedomostí Cognee je stále k dispozícii. Agent môže načítať tie isté dlhodobé znalosti v úplne novom rozhovore.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zhrnutie

V tomto notebooku ste vytvorili asistenčný nástroj na programovanie, ktorý kombinuje **pracovnú pamäť MAF** (`agent.create_session()`) s **dlhodobým znalostným grafom Cognee**.

### Čo ste sa naučili
1. **Konštrukcia znalostného grafu**: Cognee spracováva nestruktúrovaný text a vytvára graf + vektorovú pamäť.
2. **Obohacovanie grafu pomocou memify**: Odvodené fakty a bohatšie vzťahy nad vaším existujúcim grafom.
3. **Integrácia MAF + Cognee**: Funkcie `@tool` umožňujú agentovi MAF prirodzene dotazovať sa na graf Cognee.
4. **Pracovná pamäť + dlhodobá pamäť**: `AgentSession` (cez `agent.create_session()`) poskytuje kontext relácie, zatiaľ čo Cognee poskytuje trvalé znalosti.
5. **Filtrované vyhľadávanie pomocou NodeSets**: Cielenie na konkrétne podmnožiny znalostného grafu (napr. iba princípy).

### Kľúčové zistenia
- **Cognee** premieňa surový text na štruktúrovanú pamäť s uvedomením vzťahov — silnejšie než ploché vektorové úložisko.
- **Funkcie `@tool`** vytvárajú čisté prepojenie medzi agentmi MAF a externými znalosťovými systémami.
- **`AgentSession`** (cez `agent.create_session()`) udržiava kontext každej konverzácie oddelene od dlhodobo uložených znalostí.
- Rovnaký znalostný graf slúži viacerým reláciám a agentom.

### Reálne použitia
- **Vývojárski kopiloti**: kontrola kódu, analýza incidentov, asistent architektúry
- **Kopiloti pre zákazníkov**: podpora nad dokumentáciou produktov, FAQ a poznámkami CRM
- **Interní expertní kopiloti**: asistenti pre pravidlá, právnu alebo bezpečnostnú oblasť na základe smerníc
- **Zjednotené dátové vrstvy**: kombinovanie štruktúrovaných a nestruktúrovaných údajov do jedného dotazovateľného grafu

### Ďalšie kroky
- Experimentovať s časovou uvedomelosťou v Cognee
- Definovať OWL ontológiu pre doménovo špecifickú kvalitu grafu
- Pridať spätnoväzobné slučky používateľa na zlepšenie vyhľadávania v čase
- Škálovať na multi-agentné systémy zdieľajúce rovnakú pamäťovú vrstvu Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornenie**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď usilovne dbáme na presnosť, berte prosím na vedomie, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho rodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
